# Bay Area Traffic Accident Analysis
## Highway 101 & 237: Palo Alto to San Jose/Milpitas/Fremont

Analyzing traffic accident patterns using SWITRS data from UC Berkeley TIMS.

**Data Source:** [TIMS (Transportation Injury Mapping System)](https://tims.berkeley.edu/) - UC Berkeley SafeTREC

In [19]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium.plugins import HeatMap
from pathlib import Path

# Output directory for charts
CHARTS_DIR = Path('../charts')
CHARTS_DIR.mkdir(exist_ok=True)

# Plotly template
TEMPLATE = 'plotly_white'

## 1. Load & Prepare Data

In [20]:
# Load Santa Clara file
df = pd.read_csv('data/Crashes-SantaClara-01012019-09302025.csv', low_memory=False)

df['COUNTY'] = 'Santa Clara'

print(f'Santa Clara: {len(df):,}')

Santa Clara: 42,529


In [21]:
# Parse datetime
df['COLLISION_DATE'] = pd.to_datetime(df['COLLISION_DATE'], errors='coerce')
df['COLLISION_TIME'] = df['COLLISION_TIME'].astype(str).str.zfill(4)
df['hour'] = pd.to_numeric(df['COLLISION_TIME'].str[:2], errors='coerce').fillna(0).astype(int)
df['minute'] = pd.to_numeric(df['COLLISION_TIME'].str[2:4], errors='coerce').fillna(0).astype(int)

# Date components
df['day_of_week'] = df['COLLISION_DATE'].dt.day_name()
df['day_num'] = df['COLLISION_DATE'].dt.dayofweek
df['month'] = df['COLLISION_DATE'].dt.month
df['year'] = df['COLLISION_DATE'].dt.year
df['month_name'] = df['COLLISION_DATE'].dt.month_name()
df['is_weekend'] = df['day_num'] >= 5

# Severity labels
severity_map = {1: 'Fatal', 2: 'Severe Injury', 3: 'Visible Injury', 4: 'Complaint of Pain', 0: 'Property Damage Only'}
df['severity_label'] = df['COLLISION_SEVERITY'].map(severity_map).fillna('Unknown')

print(f'Date range: {df["COLLISION_DATE"].min().date()} to {df["COLLISION_DATE"].max().date()}')

Date range: 2019-01-01 to 2025-09-30


In [22]:
# Filter for 101 and 237 routes
# Check what route column exists
print('Columns containing route info:')
route_cols = [c for c in df.columns if 'ROUTE' in c.upper() or 'RD' in c.upper()]
print(route_cols)

# Check PRIMARY_RD for highway mentions
print('\nSample highway mentions in PRIMARY_RD:')
hwy_filter = df['PRIMARY_RD'].str.contains('101|237|880', case=False, na=False)
print(df[hwy_filter]['PRIMARY_RD'].value_counts().head(20))

Columns containing route info:
['PRIMARY_RD', 'SECONDARY_RD', 'STATE_ROUTE', 'ROUTE_SUFFIX']

Sample highway mentions in PRIMARY_RD:
PRIMARY_RD
US-101 S/B                 1585
US-101 N/B                 1538
I-880 S/B                   625
I-880 N/B                   416
SR-237 W/B                  301
US-101 SOUTHBOUND           298
SR-237 E/B                  278
US101 S/B                   247
US101 N/B                   246
US-101 NORTHBOUND           202
US-101 NB                   118
US 101 N/B                   83
US-101 SB                    83
US 101 S/B                   63
RT 237                       62
US-101                       35
RT 101                       25
I-280 S/B TO US-101 S/B      17
I-880 SOUTHBOUND             13
I-880 N/B TO US-101 N/B      13
Name: count, dtype: int64


In [23]:
# Create filtered dataset for highway analysis
# Adjust filter based on actual data
df_hwy = df[df['PRIMARY_RD'].str.contains('101|237', case=False, na=False)].copy()
print(f'Accidents on 101/237: {len(df_hwy):,} ({len(df_hwy)/len(df)*100:.1f}% of total)')

# Use all data if highway filter is too restrictive
if len(df_hwy) < 500:
    print('Highway filter too restrictive, using all data')
    df_hwy = df.copy()

Accidents on 101/237: 6,034 (14.2% of total)


---
## 2. The "Tech Commuter" Pattern

**Hypothesis:** Weekday accidents show distinct AM/PM rush hour peaks, while weekends have a completely different distribution.

In [24]:
# Weekday vs Weekend pattern
df_hwy['day_type'] = df_hwy['is_weekend'].map({True: 'Weekend', False: 'Weekday'})

hourly_pattern = df_hwy.groupby(['hour', 'day_type']).size().reset_index(name='count')

fig = px.line(hourly_pattern, x='hour', y='count', color='day_type',
              title='<b>The Tech Commuter Pattern</b><br><sup>Distinct AM/PM peaks on weekdays vs flat weekend distribution</sup>',
              labels={'count': 'Accidents', 'hour': 'Hour of Day', 'day_type': ''},
              color_discrete_map={'Weekday': '#e74c3c', 'Weekend': '#3498db'},
              template=TEMPLATE)

fig.update_traces(line=dict(width=3), mode='lines+markers')
fig.update_layout(xaxis=dict(dtick=2), hovermode='x unified')

# Add rush hour annotations
fig.add_vrect(x0=7, x1=9, fillcolor='red', opacity=0.1, line_width=0, annotation_text='AM Rush')
fig.add_vrect(x0=16, x1=19, fillcolor='orange', opacity=0.1, line_width=0, annotation_text='PM Rush')

fig.write_html(CHARTS_DIR / 'commuter_pattern.html', include_plotlyjs='cdn')
fig.show()

In [25]:
# Return to Office impact: Compare 2021-2022 (WFH) vs 2023-2024 (RTO)
df_hwy['period'] = df_hwy['year'].apply(
    lambda y: 'Pre-COVID (2019)' if y == 2019 
    else 'COVID/WFH (2020-2022)' if y in [2020, 2021, 2022]
    else 'RTO Era (2023+)' if y >= 2023 else 'Other'
)

period_hourly = df_hwy[df_hwy['period'] != 'Other'].groupby(['hour', 'period']).size().reset_index(name='count')

fig = px.line(period_hourly, x='hour', y='count', color='period',
              title='<b>Return to Office Impact</b><br><sup>How did RTO mandates affect rush hour accidents?</sup>',
              labels={'count': 'Accidents', 'hour': 'Hour'},
              template=TEMPLATE)

fig.update_traces(line=dict(width=2.5))
fig.update_layout(xaxis=dict(dtick=2))

fig.write_html(CHARTS_DIR / 'rto_impact.html', include_plotlyjs='cdn')
fig.show()

---
## 3. The SR-237 "Sun Glare" Effect

**Hypothesis:** SR-237 runs East-West. Morning sun (eastbound) and evening sun (westbound) cause vision-obscured accidents, especially during equinox months (March, September).

In [26]:
# Check for vision/lighting columns
vision_cols = [c for c in df.columns if 'LIGHT' in c.upper() or 'VISION' in c.upper() or 'WEATHER' in c.upper()]
print('Vision/Lighting columns:', vision_cols)

Vision/Lighting columns: ['CITY_DIVISION_LAPD', 'WEATHER_1', 'WEATHER_2', 'LIGHTING']


In [27]:
# Analyze lighting conditions
if 'LIGHTING' in df_hwy.columns:
    lighting_map = {
        'A': 'Daylight',
        'B': 'Dusk/Dawn',
        'C': 'Dark - Street Lights',
        'D': 'Dark - No Lights',
        'E': 'Dark - Lights Not Working'
    }
    df_hwy['lighting_label'] = df_hwy['LIGHTING'].map(lighting_map).fillna('Unknown')
    
    # Accidents by hour and lighting
    lighting_hour = df_hwy.groupby(['hour', 'lighting_label']).size().reset_index(name='count')
    
    fig = px.area(lighting_hour, x='hour', y='count', color='lighting_label',
                  title='<b>Lighting Conditions Throughout the Day</b>',
                  labels={'count': 'Accidents', 'hour': 'Hour'},
                  template=TEMPLATE)
    
    fig.update_layout(xaxis=dict(dtick=2))
    fig.write_html(CHARTS_DIR / 'lighting_conditions.html', include_plotlyjs='cdn')
    fig.show()

In [28]:
# Sun glare hypothesis: AM (6-9) and PM (16-19) during equinox months
equinox_months = [3, 4, 9, 10]  # March, April, September, October
df_hwy['is_equinox'] = df_hwy['month'].isin(equinox_months)
df_hwy['is_glare_hour'] = df_hwy['hour'].isin([6, 7, 8, 16, 17, 18, 19])

# Compare equinox vs non-equinox for glare hours
glare_analysis = df_hwy[df_hwy['is_glare_hour']].groupby(['hour', 'is_equinox']).size().reset_index(name='count')
glare_analysis['period'] = glare_analysis['is_equinox'].map({True: 'Equinox Months (Mar/Apr/Sep/Oct)', False: 'Other Months'})

fig = px.bar(glare_analysis, x='hour', y='count', color='period', barmode='group',
             title='<b>Sun Glare Hypothesis</b><br><sup>Do equinox months have more morning/evening accidents?</sup>',
             labels={'count': 'Accidents', 'hour': 'Hour'},
             template=TEMPLATE)

fig.write_html(CHARTS_DIR / 'sun_glare.html', include_plotlyjs='cdn')
fig.show()

---
## 4. Heatmap: Day vs Hour

In [29]:
# Interactive Heatmap
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot = df_hwy.groupby(['day_of_week', 'hour']).size().reset_index(name='count')
heatmap_data = pivot.pivot(index='day_of_week', columns='hour', values='count').fillna(0)
heatmap_data = heatmap_data.reindex(day_order)

fig = px.imshow(heatmap_data,
                labels=dict(x='Hour of Day', y='Day of Week', color='Accidents'),
                title='<b>When Do Crashes Happen?</b><br><sup>Interactive heatmap - hover for details</sup>',
                color_continuous_scale='YlOrRd',
                aspect='auto',
                template=TEMPLATE)

fig.update_xaxes(side='bottom', dtick=1)
fig.update_traces(hovertemplate='%{y}, %{x}:00<br><b>%{z:,} accidents</b><extra></extra>')

fig.write_html(CHARTS_DIR / 'heatmap.html', include_plotlyjs='cdn')
fig.show()

---
## 5. Severity Analysis

In [30]:
# Severity donut chart
severity_order = ['Fatal', 'Severe Injury', 'Visible Injury', 'Complaint of Pain', 'Property Damage Only']
severity_counts = df_hwy['severity_label'].value_counts().reset_index()
severity_counts.columns = ['severity', 'count']

colors = ['#c0392b', '#e74c3c', '#e67e22', '#f39c12', '#27ae60']

fig = px.pie(severity_counts, values='count', names='severity',
             title='<b>Accident Severity Breakdown</b>',
             color_discrete_sequence=colors,
             hole=0.4,
             template=TEMPLATE)

fig.update_traces(textposition='outside', textinfo='percent+label')
fig.write_html(CHARTS_DIR / 'severity.html', include_plotlyjs='cdn')
fig.show()

In [31]:
# Are late-night accidents more severe?
df_hwy['time_bucket'] = pd.cut(df_hwy['hour'], 
                                bins=[-1, 6, 10, 15, 19, 24],
                                labels=['Night (12-6AM)', 'Morning (6-10AM)', 'Midday (10AM-3PM)', 'Evening (3-7PM)', 'Night (7PM-12)'])

severity_time = df_hwy.groupby(['time_bucket', 'severity_label']).size().reset_index(name='count')

# Calculate percentage within each time bucket
totals = severity_time.groupby('time_bucket')['count'].transform('sum')
severity_time['pct'] = severity_time['count'] / totals * 100

fig = px.bar(severity_time, x='time_bucket', y='pct', color='severity_label',
             title='<b>Are Late-Night Accidents More Severe?</b><br><sup>Severity distribution by time of day</sup>',
             labels={'pct': 'Percentage', 'time_bucket': 'Time of Day'},
             color_discrete_map={'Fatal': '#c0392b', 'Severe Injury': '#e74c3c', 
                                 'Visible Injury': '#e67e22', 'Complaint of Pain': '#f39c12',
                                 'Property Damage Only': '#27ae60'},
             template=TEMPLATE)

fig.update_layout(barmode='stack')
fig.write_html(CHARTS_DIR / 'severity_by_time.html', include_plotlyjs='cdn')
fig.show()

---
## 6. Year-over-Year Trends

In [32]:
# Annual trend with COVID annotation
yearly = df_hwy.groupby(['year', 'COUNTY']).size().reset_index(name='count')

fig = px.line(yearly, x='year', y='count', color='COUNTY',
              title='<b>Traffic Accidents Over Time</b><br><sup>Notice the COVID-19 impact in 2020 and post-RTO surge</sup>',
              labels={'count': 'Accidents', 'year': 'Year'},
              markers=True,
              template=TEMPLATE)

fig.add_annotation(x=2020, y=yearly[yearly['year']==2020]['count'].max() * 0.8,
                   text='COVID-19<br>Lockdowns', showarrow=True, arrowhead=2)

fig.update_traces(line=dict(width=3), marker=dict(size=10))
fig.write_html(CHARTS_DIR / 'yearly_trend.html', include_plotlyjs='cdn')
fig.show()

---
## 7. Hotspot Analysis: The "Weaving Zones"

In [33]:
# Top accident locations
df_hwy['location'] = df_hwy['PRIMARY_RD'].fillna('') + ' @ ' + df_hwy['SECONDARY_RD'].fillna('')
df_hwy['location'] = df_hwy['location'].str.strip(' @ ')

top_locs = df_hwy.groupby('location').size().reset_index(name='count')
top_locs = top_locs.sort_values('count', ascending=False).head(15)

fig = px.bar(top_locs, y='location', x='count', orientation='h',
             title='<b>Top 15 Accident Hotspots</b><br><sup>Where do crashes cluster?</sup>',
             labels={'count': 'Accidents', 'location': ''},
             color='count',
             color_continuous_scale='Reds',
             template=TEMPLATE)

fig.update_layout(yaxis=dict(autorange='reversed'), coloraxis_showscale=False, height=500)
fig.write_html(CHARTS_DIR / 'hotspots.html', include_plotlyjs='cdn')
fig.show()

---
## 8. Interactive Map

In [34]:
# Find coordinate columns
lat_col = next((c for c in df_hwy.columns if 'LATITUDE' in c.upper() or c == 'POINT_Y'), None)
lon_col = next((c for c in df_hwy.columns if 'LONGITUDE' in c.upper() or c == 'POINT_X'), None)

print(f'Latitude column: {lat_col}')
print(f'Longitude column: {lon_col}')

Latitude column: LATITUDE
Longitude column: LONGITUDE


In [35]:
if lat_col and lon_col:
    # Filter valid coordinates in Bay Area bounds
    df_geo = df_hwy.dropna(subset=[lat_col, lon_col])
    df_geo = df_geo[(df_geo[lat_col] > 37.0) & (df_geo[lat_col] < 37.8)]
    df_geo = df_geo[(df_geo[lon_col] > -122.5) & (df_geo[lon_col] < -121.5)]
    
    print(f'Creating map with {len(df_geo):,} geocoded accidents...')
    
    # Create map
    m = folium.Map(location=[37.4, -121.95], zoom_start=11, tiles='cartodbpositron')
    
    # Sample for performance
    sample_size = min(15000, len(df_geo))
    sample = df_geo.sample(sample_size, random_state=42)
    
    # Add heatmap
    heat_data = sample[[lat_col, lon_col]].values.tolist()
    HeatMap(heat_data, radius=10, blur=15, max_zoom=13, gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'}).add_to(m)
    
    # Add title
    title_html = '''<div style="position: fixed; top: 10px; left: 50px; z-index: 9999; 
                    background-color: white; padding: 10px; border-radius: 5px; box-shadow: 0 2px 5px rgba(0,0,0,0.2);">
                    <b>Bay Area Traffic Accident Heatmap</b><br>
                    <small>101 & 237 Corridor (2019-2025)</small></div>'''
    m.get_root().html.add_child(folium.Element(title_html))
    
    m.save(CHARTS_DIR / 'accident_map.html')
    print('Map saved!')
    m
else:
    print('No coordinate columns found')

Creating map with 5,626 geocoded accidents...
Map saved!


---
## 9. Key Statistics for Blog Post

In [36]:
print('=' * 50)
print('KEY FINDINGS FOR BLOG POST')
print('=' * 50)

print(f'\n📊 DATASET OVERVIEW')
print(f'   Total accidents analyzed: {len(df_hwy):,}')
print(f'   Date range: {df_hwy["COLLISION_DATE"].min().date()} to {df_hwy["COLLISION_DATE"].max().date()}')
print(f'   Alameda County: {len(df_hwy[df_hwy["COUNTY"]=="Alameda"]):,}')
print(f'   Santa Clara County: {len(df_hwy[df_hwy["COUNTY"]=="Santa Clara"]):,}')

print(f'\n⏰ TIME PATTERNS')
peak_hour = df_hwy.groupby('hour').size().idxmax()
print(f'   Peak hour: {peak_hour}:00 ({df_hwy[df_hwy["hour"]==peak_hour].shape[0]:,} accidents)')
peak_day = df_hwy.groupby('day_of_week').size().idxmax()
print(f'   Most dangerous day: {peak_day}')

print(f'\n💀 SEVERITY')
fatal = len(df_hwy[df_hwy['COLLISION_SEVERITY'] == 1])
severe = len(df_hwy[df_hwy['COLLISION_SEVERITY'] == 2])
print(f'   Fatal accidents: {fatal:,} ({fatal/len(df_hwy)*100:.2f}%)')
print(f'   Severe injury: {severe:,} ({severe/len(df_hwy)*100:.2f}%)')

print(f'\n📉 COVID IMPACT')
y2019 = len(df_hwy[df_hwy['year']==2019])
y2020 = len(df_hwy[df_hwy['year']==2020])
covid_drop = (y2019 - y2020) / y2019 * 100 if y2019 > 0 else 0
print(f'   2019: {y2019:,} accidents')
print(f'   2020: {y2020:,} accidents')
print(f'   COVID drop: {covid_drop:.1f}%')

print(f'\n📍 TOP HOTSPOT')
top_loc = df_hwy.groupby('location').size().idxmax()
print(f'   {top_loc}: {df_hwy.groupby("location").size().max():,} accidents')

KEY FINDINGS FOR BLOG POST

📊 DATASET OVERVIEW
   Total accidents analyzed: 6,034
   Date range: 2019-01-01 to 2025-09-30
   Alameda County: 0
   Santa Clara County: 6,034

⏰ TIME PATTERNS
   Peak hour: 17:00 (511 accidents)
   Most dangerous day: Friday

💀 SEVERITY
   Fatal accidents: 134 (2.22%)
   Severe injury: 388 (6.43%)

📉 COVID IMPACT
   2019: 1,077 accidents
   2020: 677 accidents
   COVID drop: 37.1%

📍 TOP HOTSPOT
   US-101 S/B @ SAN ANTONIO ROAD: 110 accidents


---
## Charts Generated

All charts saved to `../charts/`:

1. `commuter_pattern.html` - Weekday vs Weekend hourly patterns
2. `rto_impact.html` - Return to Office impact analysis
3. `sun_glare.html` - Equinox sun glare hypothesis
4. `heatmap.html` - Day/hour interactive heatmap
5. `severity.html` - Severity donut chart
6. `severity_by_time.html` - Are late-night accidents worse?
7. `yearly_trend.html` - Year-over-year with COVID annotation
8. `hotspots.html` - Top 15 dangerous locations
9. `accident_map.html` - Interactive folium heatmap

In [ ]:
# 10. Verification: Peak Commute Probability & RTO Trends

# Define peak hours: 7-10 AM and 3-7 PM
peak_hours = [7, 8, 9, 15, 16, 17, 18]

# Calculate daily stats
# Group by date to check if a specific day had a peak-hour accident
daily_stats = df_hwy.groupby(df_hwy['COLLISION_DATE'].dt.date).apply(
    lambda x: x['hour'].isin(peak_hours).any()
)

total_accident_days = len(daily_stats)
peak_accident_days = daily_stats.sum()

print(f"Days with at least one accident on 101/237: {total_accident_days}")
print(f"Days with at least one PEAK accident: {peak_accident_days}")
print(f"Probability (on days with accidents): {peak_accident_days/total_accident_days:.1%}")

# Yearly RTO Check
print("\nYearly Accident Counts (Santa Clara 101/237):")
yearly = df_hwy.groupby('year').size()
print(yearly)

# Compare 2019 baseline vs 2023
if 2019 in yearly.index and 2023 in yearly.index:
    diff = (yearly[2023] - yearly[2019]) / yearly[2019]
    print(f"\n2019 vs 2023 Change: {diff:.1%}")